# Energy-Aware IK-Assisted DQN (Teacher → Distillation → ONNX)

Run in **Google Colab** (GPU) for fast training. Outputs: teacher.pt, student.pt, student.onnx, CSVs, plots.

In [ ]:
# Upload the matching project source ZIP generated beside this notebook.
from google.colab import files
import os, zipfile, io
from pathlib import Path
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith('.zip')), None)
if zip_name is None:
    raise ValueError('Upload colab_source.zip from the project folder.')
PROJECT_DIR = '/content/energy_aware_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
project_root = Path(PROJECT_DIR).resolve()
with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
    for member in archive.infolist():
        normalized_name = member.filename.replace('\\', '/')
        target = (project_root / normalized_name).resolve()
        if project_root not in target.parents and target != project_root:
            raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        if member.is_dir():
            target.mkdir(parents=True, exist_ok=True)
        else:
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(member) as source, target.open('wb') as destination:
                destination.write(source.read())
requirements_files = list(project_root.rglob('requirements.txt'))
if not requirements_files:
    raise FileNotFoundError('The uploaded ZIP has no requirements.txt. Upload the latest colab_source.zip.')
PROJECT_ROOT = min(requirements_files, key=lambda p: len(p.parts)).parent
os.chdir(PROJECT_ROOT)
print(f'Project files loaded from: {PROJECT_ROOT}')
!pip install -r requirements.txt -q
!pip install onnx onnxscript onnxruntime -q

import pybullet as p
import torch
import gymnasium
print(f"PyBullet OK, PyTorch {torch.__version__}, Gymnasium {gymnasium.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Extract the source ZIP from its uploaded bytes, independent of Colab's working directory.
import sys, os, zipfile, io
from pathlib import Path
from google.colab import files
project_dir = Path('/content/energy_aware_project')
project_dir.mkdir(parents=True, exist_ok=True)
source_dir = next((p.parent for p in project_dir.rglob('environment.py')
                   if p.parent.name == 'src' and (p.parent / 'dqn_teacher.py').is_file()), None)
if source_dir is None:
    print('Upload colab_source.zip now.')
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No ZIP was uploaded. Choose colab_source.zip.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_names[0]])) as archive:
        zip_members = archive.namelist()
        for member in archive.infolist():
            normalized_name = member.filename.replace('\\', '/')
            target = (project_dir / normalized_name).resolve()
            if project_dir.resolve() not in target.parents and target != project_dir.resolve():
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, target.open('wb') as destination:
                    destination.write(source.read())
    source_dir = next((p.parent for p in project_dir.rglob('environment.py')
                       if p.parent.name == 'src' and (p.parent / 'dqn_teacher.py').is_file()), None)
    if source_dir is None:
        print('ZIP entries containing source files:', [n for n in zip_members if n.endswith(('environment.py', 'dqn_teacher.py'))])
        raise FileNotFoundError('ZIP did not extract the expected src/environment.py and src/dqn_teacher.py files.')
sys.path.insert(0, str(source_dir))

from environment import RobotArmEnv
from dqn_teacher import DQNAgent, QNetwork
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

In [ ]:
# ============================================================
# 1. IK-DEMONSTRATION WARM-UP + DQN FINE-TUNING (500 episodes)
# ============================================================
env = RobotArmEnv(render_mode=None)
teacher = DQNAgent(state_dim=23, num_joints=6, device=device)

# Expert demonstrations make target-reaching learnable before energy fine-tuning.
import pybullet as p
action_lookup = {tuple(a.tolist()): i for i, a in enumerate(teacher.action_table)}
expert_states, expert_actions = [], []
DEMO_EPISODES = 100
demo_successes = 0
for demo_ep in range(DEMO_EPISODES):
    demo_obs, _ = env.reset(seed=50_000 + demo_ep)
    demo_terminated = demo_truncated = False
    while not (demo_terminated or demo_truncated):
        target_angles = p.calculateInverseKinematics(
            env.robot_id, env.end_effector_link, env.target_position.tolist()
        )
        demo_action = np.asarray([
            np.sign(target_angles[j] - p.getJointState(env.robot_id, j)[0])
            for j in env.controlled_joints
        ], dtype=np.float32)
        demo_idx = action_lookup[tuple(demo_action.tolist())]
        expert_states.append(demo_obs.copy())
        expert_actions.append(demo_idx)
        demo_obs, _, demo_terminated, demo_truncated, demo_info = env.step(demo_action)
    demo_successes += int(bool(demo_info['task_done']))
expert_data = (np.asarray(expert_states, dtype=np.float32),
               np.asarray(expert_actions, dtype=np.int64))
print(f'IK demonstrations: {demo_successes}/{DEMO_EPISODES} completed, {len(expert_states)} steps')
for bc_step in range(3000):
    batch_ids = np.random.randint(0, len(expert_states), size=256)
    states_t = torch.as_tensor(expert_data[0][batch_ids], device=device)
    actions_t = torch.as_tensor(expert_data[1][batch_ids], device=device)
    bc_loss = F.cross_entropy(teacher.q_network(states_t), actions_t)
    teacher.optimizer.zero_grad()
    bc_loss.backward()
    teacher.optimizer.step()
teacher.update_target_network()
bc_state = {k: v.detach().cpu().clone() for k, v in teacher.q_network.state_dict().items()}
print(f'IK behavior-cloning warm-up loss: {bc_loss.item():.4f}')
# Check the warm-start policy on held-out seeds before DQN fine-tuning.
bc_validation_success = 0
for val_ep in range(30):
    val_obs, _ = env.reset(seed=60_000 + val_ep)
    val_terminated = val_truncated = False
    while not (val_terminated or val_truncated):
        val_idx, val_action = teacher.select_action(val_obs, epsilon=0.0)
        val_obs, _, val_terminated, val_truncated, val_info = env.step(val_action)
    bc_validation_success += int(bool(val_info['task_done']))
print(f'IK-assisted DQN warm-start validation: {bc_validation_success}/30 completed')

EPISODES = 500
history = []

for episode in range(1, EPISODES + 1):
    obs, _ = env.reset(seed=episode)
    episode_reward = 0.0
    episode_losses = []
    terminated = False
    truncated = False
    epsilon = max(0.01, 0.10 * (0.995 ** (episode - 1)))
    
    while not (terminated or truncated):
        action_idx, action = teacher.select_action(obs, epsilon)
        next_obs, reward, terminated, truncated, info = env.step(action)
        teacher.store(obs, action_idx, reward, next_obs, terminated)
        loss = teacher.learn(batch_size=64, expert_data=expert_data, imitation_weight=0.10)
        if loss is not None:
            episode_losses.append(loss)
        obs = next_obs
        episode_reward += reward
    
    # Save teacher metrics before reset(): reset clears the environment counters.
    teacher_peak = env.peak_grid_draw
    teacher_energy = info['total_grid_draw']
    teacher_buffer_used = info['total_buffer_used']
    teacher_done = bool(info['task_done'])
    teacher_distance = float(info['distance'])
    
    # Baseline comparison: same seed and same buffer model/initial charge.
    import pybullet as p
    env.buffer_enabled = True
    obs_b, _ = env.reset(seed=episode)
    baseline_total = 0.0
    baseline_peak = 0.0
    t_done = False
    t_trunc = False
    while not (t_done or t_trunc):
        target_angles = p.calculateInverseKinematics(
            env.robot_id, env.end_effector_link, env.target_position.tolist()
        )
        action = np.asarray([
            np.sign(target_angles[j] - p.getJointState(env.robot_id, j)[0])
            for j in env.controlled_joints
        ], dtype=np.float32)
        obs_b, _, t_done, t_trunc, info_b = env.step(action)
        baseline_total += info_b['grid_draw']
        baseline_peak = max(baseline_peak, info_b['grid_draw'])
    env.buffer_enabled = True
    
    baseline_done = bool(info_b['task_done'])
    comparison_valid = teacher_done and baseline_done
    peak_reduction = ((baseline_peak - teacher_peak) / baseline_peak * 100
                      if comparison_valid and baseline_peak > 0 else np.nan)
    
    result = {
        'episode': episode,
        'reward': round(episode_reward, 4),
        'grid_energy': round(teacher_energy, 4),
        'baseline_grid_energy': round(baseline_total, 4),
        'peak_grid_draw': round(teacher_peak, 4),
        'baseline_peak_grid_draw': round(baseline_peak, 4),
        'peak_reduction_percent': round(peak_reduction, 2) if comparison_valid else np.nan,
        'energy_comparison_valid': comparison_valid,
        'baseline_task_done': baseline_done,
        'task_done': teacher_done,
        'distance': round(teacher_distance, 4),
        'buffer_used': round(teacher_buffer_used, 4),
        'epsilon': round(epsilon, 4),
        'mean_loss': round(float(np.mean(episode_losses)), 6) if episode_losses else None,
    }
    history.append(result)
    
    if episode % 5 == 0:
        teacher.update_target_network()
    
    if episode % 20 == 0:
        done_so_far = sum(1 for h in history if h['task_done'])
        print(f"Ep {episode}/{EPISODES}: reward={result['reward']:.1f}, teacher={result['grid_energy']}Ws, baseline={result['baseline_grid_energy']}Ws, peak↓={result['peak_reduction_percent']:.1f}%, done={result['task_done']} | Completed: {done_so_far}/{episode}")

# Select the checkpoint with more successful held-out episodes; compare energy only on successes.
finetuned_state = {k: v.detach().cpu().clone() for k, v in teacher.q_network.state_dict().items()}
def score_checkpoint(state_dict, first_seed, count=20):
    teacher.q_network.load_state_dict(state_dict)
    successes, successful_energies = 0, []
    for val_ep in range(count):
        val_obs, _ = env.reset(seed=first_seed + val_ep)
        val_terminated = val_truncated = False
        while not (val_terminated or val_truncated):
            val_idx, val_action = teacher.select_action(val_obs, epsilon=0.0)
            val_obs, _, val_terminated, val_truncated, val_info = env.step(val_action)
        if val_info['task_done']:
            successes += 1
            successful_energies.append(val_info['total_grid_draw'])
    mean_success_energy = float(np.mean(successful_energies)) if successful_energies else float('inf')
    return successes, mean_success_energy
bc_score = score_checkpoint(bc_state, 70_000)
rl_score = score_checkpoint(finetuned_state, 70_000)
if rl_score[0] < bc_score[0] or (rl_score[0] == bc_score[0] and rl_score[1] > bc_score[1]):
    teacher.q_network.load_state_dict(bc_state)
    selected_checkpoint = 'IK demonstration warm-start'
else:
    teacher.q_network.load_state_dict(finetuned_state)
    selected_checkpoint = 'DQN fine-tuned'
teacher.update_target_network()
print(f'Checkpoint selection on held-out seeds: warm-start={bc_score}, fine-tuned={rl_score}; selected={selected_checkpoint}')
env.close()

# Save teacher
os.makedirs('models', exist_ok=True)
teacher.save('models/teacher.pt')
print("Teacher saved to models/teacher.pt")

In [ ]:
# ============================================================
# 2. KNOWLEDGE DISTILLATION (Teacher → Student)
# ============================================================

class StudentNet(nn.Module):
    """Smaller student with the teacher's discrete action set."""
    def __init__(self, state_dim, num_actions, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, num_actions)
        )
    def forward(self, x):
        return self.net(x)

# Load teacher
teacher_net = QNetwork(23, teacher.num_actions, 256).to(device)
teacher_net.load_state_dict(torch.load('models/teacher.pt', map_location=device))
teacher_net.eval()

# Create student
student = StudentNet(23, teacher.num_actions, 64).to(device)
optimizer = torch.optim.Adam(student.parameters(), lr=1e-3)

# Distillation: KL divergence with temperature
T = 3.0
alpha = 0.7

# Generate soft targets from teacher replay buffer
replay = list(teacher.replay_buffer.buffer)
states = torch.from_numpy(np.asarray([item[0] for item in replay], dtype=np.float32)).to(device)

print(f"Distilling from {len(states)} teacher experiences...")

BATCH = 256
EPOCHS = 50
distill_losses = []

for epoch in range(EPOCHS):
    idx = torch.randperm(len(states))[:BATCH]
    batch_states = states[idx]
    
    with torch.no_grad():
        teacher_logits = teacher_net(batch_states) / T
        teacher_probs = F.softmax(teacher_logits, dim=1)
    
    student_logits = student(batch_states) / T
    student_log_probs = F.log_softmax(student_logits, dim=1)
    
    kl_loss = F.kl_div(student_log_probs, teacher_probs, reduction='batchmean') * (T * T)
    
    # Also match hard targets (ground truth actions from replay)
    actions = torch.from_numpy(np.asarray([replay[i][1] for i in idx], dtype=np.int64)).to(device)
    ce_loss = F.cross_entropy(student_logits * T, actions)
    
    loss = alpha * kl_loss + (1 - alpha) * ce_loss
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    distill_losses.append(loss.item())
    if epoch % 5 == 0:
        print(f"  Epoch {epoch}/{EPOCHS}: KL={kl_loss.item():.4f}, CE={ce_loss.item():.4f}")

# Save student
torch.save(student.state_dict(), 'models/student.pt')
print("Student saved to models/student.pt")

In [ ]:
# ============================================================
# 3. MATCHED-SEED EVALUATION: IK-ASSISTED DQN, IK BASELINE, STUDENT
# ============================================================
import pandas as pd
env = RobotArmEnv(render_mode=None)

def evaluate_agent(agent, name, episodes=50):
    rewards, energies, completions, distances, peaks = [], [], [], [], []
    for ep in range(episodes):
        obs, _ = env.reset(seed=1000 + ep)
        terminated = truncated = False
        ep_reward = 0.0
        while not (terminated or truncated):
            if hasattr(agent, 'select_action'):
                action_idx, action = agent.select_action(obs, epsilon=0.0)
            else:
                # StudentNet
                with torch.no_grad():
                    logits = agent(torch.from_numpy(obs).float().unsqueeze(0).to(device))
                    action_idx = int(torch.argmax(logits, dim=1).item())
                    from dqn_teacher import build_discrete_action_table
                    action_table = build_discrete_action_table(6)
                    action = action_table[action_idx]
            obs, reward, terminated, truncated, info = env.step(action)
            ep_reward += reward
        rewards.append(ep_reward)
        energies.append(info['total_grid_draw'])
        completions.append(info['task_done'])
        distances.append(info['distance'])
        peaks.append(env.peak_grid_draw)
    print(f"{name}: Avg reward={np.mean(rewards):.1f}, Avg energy={np.mean(energies):.2f}Ws, Completion={sum(completions)}/{episodes} ({100*sum(completions)/episodes:.0f}%), Avg dist={np.mean(distances):.3f}m")
    return {'rewards': rewards, 'energies': energies, 'completions': completions, 'distances': distances, 'peaks': peaks}

def evaluate_ik_baseline(episodes=50):
    energies, completions, distances, peaks = [], [], [], []
    for ep in range(episodes):
        obs, _ = env.reset(seed=1000 + ep)
        terminated = truncated = False
        while not (terminated or truncated):
            target_angles = p.calculateInverseKinematics(env.robot_id, env.end_effector_link, env.target_position.tolist())
            action = np.asarray([np.sign(target_angles[j] - p.getJointState(env.robot_id, j)[0]) for j in env.controlled_joints], dtype=np.float32)
            obs, _, terminated, truncated, info = env.step(action)
        energies.append(info['total_grid_draw']); completions.append(bool(info['task_done']))
        distances.append(info['distance']); peaks.append(env.peak_grid_draw)
    print(f'IK baseline: Completion={sum(completions)}/{episodes} ({100*np.mean(completions):.0f}%)')
    return {'rewards': [np.nan] * episodes, 'energies': energies, 'completions': completions, 'distances': distances, 'peaks': peaks}

# Recreate teacher agent for evaluation
teacher_eval = DQNAgent(state_dim=23, num_joints=6, device=device)
teacher_eval.load('models/teacher.pt')

teacher_results = evaluate_agent(teacher_eval, "IK-assisted DQN")
baseline_results = evaluate_ik_baseline()
student_results = evaluate_agent(student, "Student")

paired_evaluation = pd.DataFrame([{
    'seed': 1000 + i,
    'teacher_task_done': bool(teacher_results['completions'][i]),
    'baseline_task_done': bool(baseline_results['completions'][i]),
    'comparison_valid': bool(teacher_results['completions'][i] and baseline_results['completions'][i]),
    'teacher_energy_ws': teacher_results['energies'][i],
    'baseline_energy_ws': baseline_results['energies'][i],
    'teacher_peak_ws': teacher_results['peaks'][i],
    'baseline_peak_ws': baseline_results['peaks'][i],
    'energy_savings_percent': ((baseline_results['energies'][i] - teacher_results['energies'][i]) / baseline_results['energies'][i] * 100) if teacher_results['completions'][i] and baseline_results['completions'][i] and baseline_results['energies'][i] > 0 else np.nan,
    'peak_reduction_percent': ((baseline_results['peaks'][i] - teacher_results['peaks'][i]) / baseline_results['peaks'][i] * 100) if teacher_results['completions'][i] and baseline_results['completions'][i] and baseline_results['peaks'][i] > 0 else np.nan,
} for i in range(50)])
paired_evaluation.to_csv('paired_evaluation.csv', index=False)

env.close()

In [ ]:
# ============================================================
# 4. ONNX EXPORT
# ============================================================
import onnx
import onnxruntime as ort

student.eval()
dummy_input = torch.randn(1, 23).to(device)

torch.onnx.export(
    student,
    dummy_input,
    'models/student.onnx',
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Verify ONNX model
onnx_model = onnx.load('models/student.onnx')
onnx.checker.check_model(onnx_model)
print("ONNX model valid")

# Test ONNX Runtime inference
ort_session = ort.InferenceSession('models/student.onnx', providers=['CPUExecutionProvider'])
ort_input = {ort_session.get_inputs()[0].name: dummy_input.cpu().numpy()}
ort_output = ort_session.run(None, ort_input)
print(f"ONNX Runtime output shape: {ort_output[0].shape}")
print("ONNX export successful → models/student.onnx")

In [ ]:
# ============================================================
# 5. EXPORT CSV REPORTS
# ============================================================
import pandas as pd
import json

# Training history
df_history = pd.DataFrame(history)
df_history.to_csv('training_history.csv', index=False)
print("Saved training_history.csv")

# Evaluation summary
def evaluation_row(name, result):
    completed = np.asarray(result['completions'], dtype=bool)
    return {'Agent': name, 'Evaluation_Episodes': len(completed), 'Completed_Episodes': int(completed.sum()),
            'Completion_Rate_%': 100*float(completed.mean()), 'Avg_Energy_All_Ws': float(np.mean(result['energies'])),
            'Avg_Energy_Completed_Ws': float(np.mean(np.asarray(result['energies'])[completed])) if completed.any() else np.nan,
            'Avg_Peak_Completed_Ws': float(np.mean(np.asarray(result['peaks'])[completed])) if completed.any() else np.nan,
            'Avg_Final_Distance_m': float(np.mean(result['distances']))}
eval_summary = pd.DataFrame([evaluation_row('IK-assisted DQN', teacher_results),
                            evaluation_row('IK baseline', baseline_results),
                            evaluation_row('Student', student_results)])
eval_summary.to_csv('evaluation_summary.csv', index=False)
print("Saved evaluation_summary.csv")

# Distillation losses
pd.DataFrame({'epoch': range(len(distill_losses)), 'loss': distill_losses}).to_csv('distillation_losses.csv', index=False)
print("Saved distillation_losses.csv")

# Model sizes
teacher_params = sum(p.numel() for p in teacher_eval.q_network.parameters())
student_params = sum(p.numel() for p in student.parameters())
pd.DataFrame({
    'Model': ['Teacher (256x256)', 'Student (64x64)'],
    'Parameters': [teacher_params, student_params],
    'Compression_Ratio': [1.0, teacher_params/student_params]
}).to_csv('model_comparison.csv', index=False)
print("Saved model_comparison.csv")

print("\n=== ALL FILES READY ===")
print("models/teacher.pt, models/student.pt, models/student.onnx")
print("training_history.csv, evaluation_summary.csv, paired_evaluation.csv, distillation_losses.csv, model_comparison.csv")

In [ ]:
# ============================================================
# 6. PLOTS FOR PRESENTATION
# ============================================================
import matplotlib.pyplot as plt

episodes = [h['episode'] for h in history]
teacher_energy = [h['grid_energy'] for h in history]
baseline_energy = [h['baseline_grid_energy'] for h in history]
peak_reduction = [h['peak_reduction_percent'] for h in history]
task_done = [h['task_done'] for h in history]
rewards = [h['reward'] for h in history]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

valid_pairs = paired_evaluation[paired_evaluation['comparison_valid']]
axes[0,0].plot(np.arange(1, len(valid_pairs)+1), valid_pairs['teacher_energy_ws'], label='IK-assisted DQN', color='#08807f')
axes[0,0].plot(np.arange(1, len(valid_pairs)+1), valid_pairs['baseline_energy_ws'], label='IK baseline', color='#e97b44')
axes[0,0].set_ylabel('Grid Energy (Ws)')
axes[0,0].set_xlabel('Matched successful test case')
axes[0,0].set_title('Energy on Matched Successful Targets')
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(np.arange(1, len(valid_pairs)+1), valid_pairs['peak_reduction_percent'], color='#08807f', marker='.')
axes[0,1].axhline(0, color='gray', linestyle='--')
axes[0,1].set_ylabel('Peak Reduction (%)')
axes[0,1].set_xlabel('Matched successful test case')
axes[0,1].set_title('Peak Reduction on Matched Successes')
axes[0,1].grid(True, alpha=0.3)

axes[0,2].plot(episodes, rewards, color='#e97b44')
axes[0,2].set_ylabel('Episode Reward')
axes[0,2].set_xlabel('Episode')
axes[0,2].set_title('Learning Curve')
axes[0,2].grid(True, alpha=0.3)

done_rate = np.convolve(task_done, np.ones(20)/20, mode='valid') * 100
axes[1,0].plot(episodes[19:], done_rate, color='#08807f')
axes[1,0].set_ylabel('Completion Rate (%)')
axes[1,0].set_xlabel('Episode')
axes[1,0].set_title('Online Training Completion (20-Episode)')
axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(distill_losses, color='#e97b44')
axes[1,1].set_ylabel('Distillation Loss')
axes[1,1].set_xlabel('Epoch')
axes[1,1].set_title('Knowledge Distillation Convergence')
axes[1,1].grid(True, alpha=0.3)

labels = ['IK-assisted DQN', 'IK baseline', 'Student']
completions = [int(np.sum(r['completions'])) for r in [teacher_results, baseline_results, student_results]]
energies = [float(eval_summary.loc[eval_summary['Agent']==name, 'Avg_Energy_Completed_Ws'].iloc[0]) for name in labels]
x = np.arange(len(labels))
width = 0.35
axes[1,2].bar(x - width/2, completions, width, label='Completed Episodes', color='#08807f')
axes[1,2].bar(x + width/2, [e/10 if np.isfinite(e) else np.nan for e in energies], width, label='Completed-task energy (Ws/10)', color='#e97b44')
axes[1,2].set_xticks(x)
axes[1,2].set_xticklabels(labels)
axes[1,2].legend()
axes[1,2].set_title('Evaluation: Completion & Energy on Successes')
axes[1,2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('presentation_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved presentation_plots.png")

In [ ]:
# ============================================================
# 7. DOWNLOAD ALL ARTIFACTS
# ============================================================
from google.colab import files
import zipfile

with zipfile.ZipFile('prototype_deliverables.zip', 'w') as z:
    for f in ['models/teacher.pt', 'models/student.pt', 'models/student.onnx',
              'training_history.csv', 'evaluation_summary.csv', 'paired_evaluation.csv',
              'distillation_losses.csv', 'model_comparison.csv',
              'presentation_plots.png']:
        if os.path.exists(f):
            z.write(f)

files.download('prototype_deliverables.zip')
print("Downloaded prototype_deliverables.zip - contains all models, CSVs, plots")